### Resources
[Building an NCAA mens basketball predictive model and quantifying its success](https://arxiv.org/abs/1412.0248)


### Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from os.path import isfile, join
import os
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

In [10]:
kaggle_data_files = [f for f in listdir('kaggle_data_sources') if isfile(join('kaggle_data_sources', f))]

for file in kaggle_data_files:
    file_path = os.path.join('kaggle_data_sources', file)

    df_name = "df_raw_" + file.replace(".csv", "")

    globals()[df_name] = pd.read_csv(file_path)

    print(f"Created DataFrame: {df_name}")

df_raw_MNCAATourneyDetailedResults

Created DataFrame: df_raw_MNCAATourneyDetailedResults
Created DataFrame: df_raw_SampleSubmissionStage2
Created DataFrame: df_raw_WSecondaryTourneyTeams
Created DataFrame: df_raw_WNCAATourneySlots
Created DataFrame: df_raw_MNCAATourneyCompactResults
Created DataFrame: df_raw_MSeasons
Created DataFrame: df_raw_SampleSubmissionStage1
Created DataFrame: df_raw_WTeams
Created DataFrame: df_raw_MRegularSeasonDetailedResults
Created DataFrame: df_raw_WNCAATourneyDetailedResults
Created DataFrame: df_raw_MNCAATourneySlots
Created DataFrame: df_raw_MGameCities
Created DataFrame: df_raw_MConferenceTourneyGames
Created DataFrame: df_raw_WNCAATourneyCompactResults
Created DataFrame: df_raw_WSecondaryTourneyCompactResults
Created DataFrame: df_raw_WSeasons
Created DataFrame: df_raw_Cities
Created DataFrame: df_raw_WRegularSeasonCompactResults
Created DataFrame: df_raw_WTeamSpellings
Created DataFrame: df_raw_WRegularSeasonDetailedResults
Created DataFrame: df_raw_MRegularSeasonCompactResults
Create

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,23,13,16,13,28,12,9,5,5,18
1445,2025,146,1222,69,1397,50,N,0,28,66,...,29,15,20,12,23,12,9,3,2,9
1446,2025,152,1196,79,1120,73,N,0,25,53,...,25,16,23,9,21,11,14,11,3,22
1447,2025,152,1222,70,1181,67,N,0,23,61,...,17,18,23,10,21,14,7,7,4,16


#### Combined Datasets

In [95]:
df_raw_MNCAATourneyDetailedResults['IsMale'] = True
df_raw_MNCAATourneyDetailedResults['IsRegularSeason'] = False

df_raw_WNCAATourneyDetailedResults['IsMale'] = False
df_raw_WNCAATourneyDetailedResults['IsRegularSeason'] = False

df_raw_MRegularSeasonDetailedResults['IsMale'] = True
df_raw_MRegularSeasonDetailedResults['IsRegularSeason'] = True

df_raw_WRegularSeasonDetailedResults['IsMale'] = False
df_raw_WRegularSeasonDetailedResults['IsRegularSeason'] = True

df_all_inclusive = pd.concat([
    df_raw_MNCAATourneyDetailedResults,
    df_raw_WNCAATourneyDetailedResults,
    df_raw_MRegularSeasonDetailedResults,
    df_raw_WRegularSeasonDetailedResults
], ignore_index=True)

df_all_inclusive.reset_index(inplace=True)

#### Enable Repeatable Feature Engineering

In [239]:
df_all_inclusive['GameID'] = df_all_inclusive.index

df_all_scorecard = df_all_inclusive[['GameID','Season','IsMale','IsRegularSeason','DayNum','NumOT','WTeamID','LTeamID','WScore','LScore','WLoc']].copy()
df_all_scorecard['IsWHome'] = df_all_scorecard['WLoc'] == 'H'
df_all_scorecard['IsLHome'] = df_all_scorecard['WLoc'] == 'A'
    

df_all_scorecard['rand_id'] = np.random.randint(0, 2, size=len(df_all_scorecard))
flip = df_all_scorecard['rand_id'] == 1

df_all_scorecard['T1_TeamID'] = np.where(flip, df_all_scorecard['WTeamID'], df_all_scorecard['LTeamID'])
df_all_scorecard['T2_TeamID'] = np.where(flip, df_all_scorecard['LTeamID'], df_all_scorecard['WTeamID'])
df_all_scorecard['T1_Score']  = np.where(flip, df_all_scorecard['WScore'],  df_all_scorecard['LScore'])
df_all_scorecard['T2_Score']  = np.where(flip, df_all_scorecard['LScore'],  df_all_scorecard['WScore'])
df_all_scorecard['T1_Home']   = np.where(flip, df_all_scorecard['IsWHome'], df_all_scorecard['IsLHome'])
df_all_scorecard['T2_Home']   = np.where(flip, df_all_scorecard['IsLHome'], df_all_scorecard['IsWHome'])

df_all_scorecard['PointDiff'] = df_all_scorecard['T1_Score'] - df_all_scorecard['T2_Score']
df_all_scorecard['T1_win'] = df_all_scorecard['T1_Score'] > df_all_scorecard['T2_Score']

df_all_scorecard.drop(columns=['WTeamID','LTeamID','WScore','LScore','rand_id'],inplace=True)


df_all_winners = df_all_inclusive[['GameID','Season','IsRegularSeason','DayNum','WTeamID','WScore','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk','WPF','LTeamID','LScore','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk','LPF']]
df_all_losers = df_all_inclusive[['GameID','Season','IsRegularSeason','DayNum','LTeamID','LScore','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk','LPF','WTeamID','WScore','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk','WPF']]

shared_cols = ['GameID','Season','IsRegularSeason','DayNum','TeamID','Score','FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','OppTeamID','OppScore','OppFGM','OppFGA','OppFGM3','OppFGA3','OppFTM','OppFTA','OppOR','OppDR','OppAst','OppTO','OppStl','OppBlk','OppPF']

df_all_winners.columns = shared_cols
df_all_losers.columns  = shared_cols

df_all_features_base = pd.concat([df_all_winners, df_all_losers], ignore_index=True)

#### Split data into experimental data and final test data

In [240]:
df_final_test_scorecards = df_all_scorecard[(df_all_scorecard['Season']>=2021)&(df_all_scorecard['IsRegularSeason']==False)]

test_game_ids = set(df_final_test_scorecards['GameID'])

df_final_test_feature_base = df_all_features_base[df_all_features_base['GameID'].isin(test_game_ids)]
df_exp_scorecards          = df_all_scorecard[~df_all_scorecard['GameID'].isin(test_game_ids)]
df_exp_feature_base        = df_all_features_base[~df_all_features_base['GameID'].isin(test_game_ids)]

In [115]:
print(df_all_scorecard.shape)
print(df_final_test_scorecards.shape)
print(df_exp_scorecards.shape)
print()
print(df_all_features_base.shape)
print(df_final_test_feature_base.shape)
print(df_exp_feature_base.shape)

(210690, 17)
(665, 17)
(210025, 17)

(421380, 33)
(1330, 33)
(420050, 33)


### Development, Training, and Testing Plan
* Baseline - The baseline is the most basic ML approach and sets a bar to exceed. It functions as the *control*.
* ML Models - ML Models will be developed to attempt at improving predictability relative to the baseline.

### Datasets
* df_all_inclusive - This includes all data from Men's and Women's regular and playoff seasons.
* df_test_final - This cannot be touched until the final moments before the final submission. This sample will be used to select which two models will be used for submission. It only includes playoff games from 2021 to current.
* df_experiment - This excludes df_test_final data and can be used for entire developmental process. This is where to create a validation test sets.

### Model Training Process
1) Each model will be built within it's own section.
2) At the end of the notebook, after all models have been developed and tested, each candidate model will be evaluated against predicting df_test_final.
3) The best two models at predicting df_test_final will be retrained using df_all_inclusive.

## Baseline Model
Linear Regression
* Target
    * T1 win
* Features
    * T1_score = T1 Score
    * T2_score = T2 Score
    * T1_fgm = T1 FGM
    * T2_fgm = T2 FGM
    * T1_fgr = T1 FGM / FGA
    * T2_fgr = T2 FGM / FGA
    * T1_fgm3 = T1 FGM3
    * T2_fgm3 = T2 FGM3
    * T1_fgr3 = T1 FGM3 / FGA3
    * T2_fgr3 = T2 FGM3 / FGA3
    * T1_ftm = T1 FTM
    * T2_ftm = T2 FTM
    * T1_ftr = T1 FTM / FTA
    * T2_ftr = T2 FTM / FTA
    * is_male
    * is_regular_season
* Season Differences
    * Regular Season - Upto current game
    * March Madness  - Overall Regular Season stats

In [225]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss


class BaselineModel:
    def __init__(self, df_scorecards, df_feature_base):
        self.df_scorecards = df_scorecards
        self.df_feature_base = df_feature_base
        self.X_cols = [
            'IsMale', 'IsRegularSeason', 'T1_avg_fgm',
            'T1_avg_fgr', 'T1_avg_fgm3', 'T1_avg_fgr3', 'T1_avg_ftm', 'T1_avg_ftr',
            'T2_avg_fgm', 'T2_avg_fgr', 'T2_avg_fgm3', 'T2_avg_fgr3', 'T2_avg_ftm',
            'T2_avg_ftr'
        ]
        self.y_col = 'T1_win'
        self.model = None
        self.df_train = None
        self.df_test = None
        self.df_baseline = None

    # ── Private helpers ────────────────────────────────────────────────────────

    def _compute_regular_season_features(self, df_feature_base):
        """Rolling per-game averages for regular season games."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()
        df_regular = df_regular.sort_values(['Season', 'TeamID', 'DayNum'])

        for col, new_col in [
            ('FGM', 'avg_fgm'), ('FGA', 'avg_fga'),
            ('FGM3', 'avg_fgm3'), ('FGA3', 'avg_fga3'),
            ('FTM', 'avg_ftm'), ('FTA', 'avg_fta')
        ]:
            df_regular[new_col] = (
                df_regular
                .groupby(['Season', 'TeamID'])[col]
                .expanding()
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

        df_regular['PrevGameID'] = (
            df_regular
            .sort_values(['TeamID', 'Season', 'DayNum'])
            .groupby(['TeamID', 'Season'])['GameID']
            .shift(-1)
        )

        df_regular['avg_fgr'] = df_regular['avg_fgm'] / df_regular['avg_fga']
        df_regular['avg_fgr3'] = df_regular['avg_fgm3'] / df_regular['avg_fga3']
        df_regular['avg_ftr'] = df_regular['avg_ftm'] / df_regular['avg_fta']

        return df_regular

    def _compute_season_summary_features(self, df_feature_base):
        """Full-season averages per team, used for March Madness matchups."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()

        df_mm = df_regular.groupby(['Season', 'TeamID']).agg(
            avg_fgm=('FGM', 'mean'),
            avg_fga=('FGA', 'mean'),
            avg_fgm3=('FGM3', 'mean'),
            avg_fga3=('FGA3', 'mean'),
            avg_ftm=('FTM', 'mean'),
            avg_fta=('FTA', 'mean')
        ).reset_index()

        df_mm['avg_fgr'] = df_mm['avg_fgm'] / df_mm['avg_fga']
        df_mm['avg_fgr3'] = df_mm['avg_fgm3'] / df_mm['avg_fga3']
        df_mm['avg_ftr'] = df_mm['avg_ftm'] / df_mm['avg_fta']

        return df_mm

    def _build_features(self, df_scorecards, df_feature_base):
        """
        Joins engineered features onto scorecards for both regular season
        and March Madness games. Returns a fully featurized dataframe.
        """
        feat_cols = ['avg_fgm', 'avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr']

        df_regular = self._compute_regular_season_features(df_feature_base)
        df_mm_features = self._compute_season_summary_features(df_feature_base)

        df_regular_features = df_regular[[
            'GameID', 'PrevGameID', 'Season', 'TeamID',
            'avg_fgm', 'avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr'
        ]].copy()

        # ── Regular season games ───────────────────────────────────────────────
        df_scorecard_reg = df_scorecards[df_scorecards['IsRegularSeason'] == True].copy()

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T1_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T2_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        # ── March Madness games ────────────────────────────────────────────────
        df_scorecard_mm = df_scorecards[df_scorecards['IsRegularSeason'] == False].copy()

        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        # ── Combine and trim columns ───────────────────────────────────────────
        keep_cols = [
            'GameID', 'IsMale', 'IsRegularSeason', 'Season', 'DayNum', 'T1_win',
            'T1_avg_fgm', 'T1_avg_fgr', 'T1_avg_fgm3', 'T1_avg_fgr3', 'T1_avg_ftm', 'T1_avg_ftr',
            'T2_avg_fgm', 'T2_avg_fgr', 'T2_avg_fgm3', 'T2_avg_fgr3', 'T2_avg_ftm', 'T2_avg_ftr'
        ]

        return pd.concat([df_scorecard_reg, df_scorecard_mm], ignore_index=True)[keep_cols]

    # ── Public interface ───────────────────────────────────────────────────────

    def get_test_data(self):
        X_test = self.df_test[self.X_cols]
        y_test = self.df_test[self.y_col]
        return X_test, y_test

    def get_model(self):
        return self.model
    
    def data_prep(self):
        """Builds features from the instance's scorecards and feature base, populates df_baseline and df_train."""
        self.df_baseline = self._build_features(self.df_scorecards, self.df_feature_base)
        self.df_train = self.df_baseline[self.df_baseline.notna().all(axis=1)]
        print('Data prepped.')

    def split_test_from_train(self):
        """
        Carves the most recent 500 March Madness games out of df_baseline
        into df_test, and removes them from df_train.
        Use this in Phase 1 when you don't have a separate test set.
        """
        df_test = (
            self.df_baseline[self.df_baseline['IsRegularSeason'] == False]
            .sort_values('Season', ascending=False)
            .iloc[0:500]
        )
        test_games = set(df_test['GameID'])
        self.df_test = df_test
        self.df_train = self.df_train[~self.df_train['GameID'].isin(test_games)]
        print(f'Test set carved: {len(df_test)} games.')

    def get_final_test_data(self, df_final_test_scorecards, df_final_test_feature_base):
        """
        Builds features from an external test set and stores in df_test.
        Features are sourced from self.df_feature_base (regular season data),
        since df_final_test_feature_base only contains March Madness games.
        Use this in Phase 2 before evaluate().
        """
        self.df_test = self._build_features(df_final_test_scorecards, self.df_feature_base)
        print('Final test data prepped.')

    def train(self, use_full_data=False):
        """
        use_full_data=False  → train on df_train only (Phases 1 & 2)
        use_full_data=True   → train on all of df_baseline (Phase 3)
        """
        source = self.df_baseline if use_full_data else self.df_train

        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]

        self.model = LogisticRegression(max_iter=1000)
        self.model.fit(X, y)
        print(f'Model trained on {"full dataset" if use_full_data else "training set"}.')

    def evaluate(self):
        """Evaluates the model against df_test."""
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')
        if self.df_test is None or self.df_test.empty:
            raise RuntimeError('No test set available. Call split_test_from_train() or get_final_test_data() first.')

        X_test = self.df_test[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y_test = self.df_test[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X_test.notna().all(axis=1) & y_test.notna()
        X_test, y_test = X_test[mask], y_test[mask]

        df_test_eval = self.df_test[mask].copy()
        df_test_eval['predicted_prob'] = self.model.predict_proba(X_test)[:, 1]

        self.brier = brier_score_loss(y_test, df_test_eval['predicted_prob'])
        print(f'Brier Score: {self.brier:.6f}')
        return self.brier

    def predict(self):
        """
        Generates March Madness win probability predictions for all hypothetical matchups.
        T1 is always the lower TeamID. Returns a dataframe with columns [ID, Pred].

        ID format: '2026_{lower_team_id}_{higher_team_id}'
        Pred: probability that the lower TeamID team wins.
        """
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')

        # Get all unique teams per gender from the March Madness scorecards
        df_mm = self.df_scorecards[self.df_scorecards['IsRegularSeason'] == False].copy()
        season = df_mm['Season'].max()  # predict for the most recent/current season

        # Build all hypothetical matchups within each gender
        records = []
        for is_male in [True, False]:
            teams = sorted(
                df_mm[df_mm['IsMale'] == is_male]['T1_TeamID'].tolist() +
                df_mm[df_mm['IsMale'] == is_male]['T2_TeamID'].tolist()
            )
            teams = sorted(set(teams))

            for i, t1 in enumerate(teams):
                for t2 in teams[i + 1:]:
                    records.append({
                        'IsMale': int(is_male),
                        'IsRegularSeason': 0,
                        'Season': season,
                        'T1_TeamID': t1,
                        'T2_TeamID': t2,
                    })

        df_matchups = pd.DataFrame(records)

        # Join season summary features (same logic as March Madness in _build_features)
        df_mm_features = self._compute_season_summary_features(self.df_feature_base)
        feat_cols = ['avg_fgm', 'avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr']

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        X = df_matchups[self.X_cols].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1)
        X = X[mask]
        df_matchups = df_matchups[mask].copy()

        df_matchups['Pred'] = self.model.predict_proba(X)[:, 1]
        df_matchups['ID'] = df_matchups.apply(
            lambda r: f"{int(r['Season'])}_{int(r['T1_TeamID'])}_{int(r['T2_TeamID'])}", axis=1
        )

        return df_matchups[['ID', 'Pred']].reset_index(drop=True)


### ── Phase workflows ────────────────────────────────────────────────────────────

### Phase 1 — Build and evaluate against a carve-out of training data
baseline = BaselineModel(df_exp_scorecards, df_exp_feature_base)
baseline.data_prep()
baseline.split_test_from_train()
baseline.train()
baseline.evaluate()
base_test_X, base_test_y = baseline.get_test_data()
base_model = baseline.get_model()

### Phase 2 — Evaluate against the official final test set
# baseline = BaselineModel(df_exp_scorecards, df_exp_feature_base)
# baseline.data_prep()
# baseline.get_final_test_data(df_final_test_scorecards, df_final_test_feature_base)
# baseline.train()
# baseline.evaluate()

### Phase 3 — Retrain on all data and generate predictions
# baseline = BaselineModel(df_all_scorecard, df_all_features_base)
# baseline.data_prep()
# baseline.train(use_full_data=True)
# baseline_prediction = baseline.predict()

Data prepped.
Test set carved: 500 games.
Model trained on training set.
Brier Score: 0.220191


### Permutation Importance

In [231]:
cols = base_test_X.columns

pred_prob_og = base_model.predict_proba(base_test_X)
brier_og = brier_score_loss(base_test_y, pred_prob_og)

print(f'Brier OG: {brier_og:.6f}')

for col in cols:
    base_test_X_shuffle = base_test_X.copy()
    base_test_X_shuffle[col] = base_test_X[col].sample(frac=1).values
    
    pred_prob = base_model.predict_proba(base_test_X_shuffle)[:, 1]
    brier = brier_score_loss(base_test_y, pred_prob)
    print(f'{col} - ' * 3)
    print(f'Brier Score: {brier:.6f}')
    print(f'Delta: {(brier_og - brier):.6f}')
    print('='* 20)
    print('='* 20)
    print()

Brier OG: 0.220191
IsMale - IsMale - IsMale - 
Brier Score: 0.220337
Delta: -0.000145

IsRegularSeason - IsRegularSeason - IsRegularSeason - 
Brier Score: 0.220191
Delta: 0.000000

T1_avg_fgm - T1_avg_fgm - T1_avg_fgm - 
Brier Score: 0.230476
Delta: -0.010285

T1_avg_fgr - T1_avg_fgr - T1_avg_fgr - 
Brier Score: 0.234785
Delta: -0.014593

T1_avg_fgm3 - T1_avg_fgm3 - T1_avg_fgm3 - 
Brier Score: 0.221582
Delta: -0.001390

T1_avg_fgr3 - T1_avg_fgr3 - T1_avg_fgr3 - 
Brier Score: 0.220653
Delta: -0.000462

T1_avg_ftm - T1_avg_ftm - T1_avg_ftm - 
Brier Score: 0.220124
Delta: 0.000067

T1_avg_ftr - T1_avg_ftr - T1_avg_ftr - 
Brier Score: 0.219714
Delta: 0.000478

T2_avg_fgm - T2_avg_fgm - T2_avg_fgm - 
Brier Score: 0.224995
Delta: -0.004804

T2_avg_fgr - T2_avg_fgr - T2_avg_fgr - 
Brier Score: 0.227810
Delta: -0.007619

T2_avg_fgm3 - T2_avg_fgm3 - T2_avg_fgm3 - 
Brier Score: 0.220093
Delta: 0.000099

T2_avg_fgr3 - T2_avg_fgr3 - T2_avg_fgr3 - 
Brier Score: 0.220709
Delta: -0.000517

T2_avg_ftm

## Inclusive Logistic Model

In [248]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss


class InclusiveLogisticModel:
    def __init__(self, df_scorecards, df_feature_base):
        self.df_scorecards = df_scorecards
        self.df_feature_base = df_feature_base
        self.X_sing_eng_cols = [
            'avg_scr','avg_fgm','avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr',
            'avg_or','avg_dr', 'avg_ast', 'avg_to', 'avg_stl', 'avg_blk', 'avg_pf',
            'opp_avg_scr','opp_avg_fgm','opp_avg_fgr', 'opp_avg_fgm3', 'opp_avg_fgr3', 'opp_avg_ftm', 'opp_avg_ftr',
            'opp_avg_or','opp_avg_dr', 'opp_avg_ast', 'opp_avg_to', 'opp_avg_stl', 'opp_avg_blk', 'opp_avg_pf']
        self.X_eng_cols = [
            'T1_avg_scr','T1_avg_fgm','T1_avg_fgr', 'T1_avg_fgm3', 'T1_avg_fgr3', 'T1_avg_ftm', 'T1_avg_ftr',
            'T1_avg_or','T1_avg_dr', 'T1_avg_ast', 'T1_avg_to', 'T1_avg_stl', 'T1_avg_blk', 'T1_avg_pf',
            'T1_opp_avg_scr','T1_opp_avg_fgm','T1_opp_avg_fgr', 'T1_opp_avg_fgm3', 'T1_opp_avg_fgr3', 'T1_opp_avg_ftm', 'T1_opp_avg_ftr',
            'T1_opp_avg_or','T1_opp_avg_dr', 'T1_opp_avg_ast', 'T1_opp_avg_to', 'T1_opp_avg_stl', 'T1_opp_avg_blk', 'T1_opp_avg_pf',
            
            'T2_avg_scr','T2_avg_fgm', 'T2_avg_fgr', 'T2_avg_fgm3', 'T2_avg_fgr3', 'T2_avg_ftm', 'T2_avg_ftr',
            'T2_avg_or','T2_avg_dr', 'T2_avg_ast', 'T2_avg_to', 'T2_avg_stl', 'T2_avg_blk', 'T2_avg_pf',
            'T2_opp_avg_scr','T2_opp_avg_fgm', 'T2_opp_avg_fgr', 'T2_opp_avg_fgm3', 'T2_opp_avg_fgr3', 'T2_opp_avg_ftm', 'T2_opp_avg_ftr',
            'T2_opp_avg_or','T2_opp_avg_dr', 'T2_opp_avg_ast', 'T2_opp_avg_to', 'T2_opp_avg_stl', 'T2_opp_avg_blk', 'T2_opp_avg_pf',
        ]
        self.X_cols = ['IsMale', 'IsRegularSeason'] + self.X_eng_cols
        self.y_col = 'T1_win'
        self.model = None
        self.df_train = None
        self.df_test = None
        self.df_baseline = None

    # ── Private helpers ────────────────────────────────────────────────────────

    def _compute_regular_season_features(self, df_feature_base):
        """Rolling per-game averages for regular season games."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()
        df_regular = df_regular.sort_values(['Season', 'TeamID', 'DayNum'])

        for col, new_col in [
            ('Score', 'avg_scr'), ('FGM', 'avg_fgm'), ('FGA', 'avg_fga'), ('FGM3', 'avg_fgm3'), 
            ('FGA3', 'avg_fga3'), ('FTM', 'avg_ftm'), ('FTA', 'avg_fta'),
            ('OR', 'avg_or'), ('DR', 'avg_dr'), ('Ast', 'avg_ast'), 
            ('TO', 'avg_to'), ('Stl', 'avg_stl'), ('Blk', 'avg_blk'), ('PF', 'avg_pf'),

            ('OppScore', 'opp_avg_scr'), ('OppFGM', 'opp_avg_fgm'), ('OppFGA', 'opp_avg_fga'), ('OppFGM3', 'opp_avg_fgm3'), 
            ('OppFGA3', 'opp_avg_fga3'), ('OppFTM', 'opp_avg_ftm'), ('OppFTA', 'opp_avg_fta'),
            ('OppOR', 'opp_avg_or'), ('OppDR', 'opp_avg_dr'), ('OppAst', 'opp_avg_ast'), 
            ('OppTO', 'opp_avg_to'), ('OppStl', 'opp_avg_stl'), ('OppBlk', 'opp_avg_blk'), ('OppPF', 'opp_avg_pf'),
        ]:
            df_regular[new_col] = (
                df_regular
                .groupby(['Season', 'TeamID'])[col]
                .expanding()
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

        df_regular['PrevGameID'] = (
            df_regular
            .sort_values(['TeamID', 'Season', 'DayNum'])
            .groupby(['TeamID', 'Season'])['GameID']
            .shift(-1)
        )

        df_regular['avg_fgr'] = df_regular['avg_fgm'] / df_regular['avg_fga']
        df_regular['avg_fgr3'] = df_regular['avg_fgm3'] / df_regular['avg_fga3']
        df_regular['avg_ftr'] = df_regular['avg_ftm'] / df_regular['avg_fta']
        df_regular['opp_avg_fgr'] = df_regular['opp_avg_fgm'] / df_regular['opp_avg_fga']
        df_regular['opp_avg_fgr3'] = df_regular['opp_avg_fgm3'] / df_regular['opp_avg_fga3']
        df_regular['opp_avg_ftr'] = df_regular['opp_avg_ftm'] / df_regular['opp_avg_fta']

        return df_regular

    def _compute_season_summary_features(self, df_feature_base):
        """Full-season averages per team, used for March Madness matchups."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()

        # Define the mapping for all base stats to be averaged
        agg_map = {
            # Team Offense
            'avg_scr': ('Score', 'mean'),
            'avg_fgm': ('FGM', 'mean'),
            'avg_fga': ('FGA', 'mean'),
            'avg_fgm3': ('FGM3', 'mean'),
            'avg_fga3': ('FGA3', 'mean'),
            'avg_ftm': ('FTM', 'mean'),
            'avg_fta': ('FTA', 'mean'),
            'avg_or': ('OR', 'mean'),
            'avg_dr': ('DR', 'mean'),
            'avg_ast': ('Ast', 'mean'),
            'avg_to': ('TO', 'mean'),
            'avg_stl': ('Stl', 'mean'),
            'avg_blk': ('Blk', 'mean'),
            'avg_pf': ('PF', 'mean'),
            
            # Team Defense
            'opp_avg_scr': ('OppScore', 'mean'),
            'opp_avg_fgm': ('OppFGM', 'mean'),
            'opp_avg_fga': ('OppFGA', 'mean'),
            'opp_avg_fgm3': ('OppFGM3', 'mean'),
            'opp_avg_fga3': ('OppFGA3', 'mean'),
            'opp_avg_ftm': ('OppFTM', 'mean'),
            'opp_avg_fta': ('OppFTA', 'mean'),
            'opp_avg_or': ('OppOR', 'mean'),
            'opp_avg_dr': ('OppDR', 'mean'),
            'opp_avg_ast': ('OppAst', 'mean'),
            'opp_avg_to': ('OppTO', 'mean'),
            'opp_avg_stl': ('OppStl', 'mean'),
            'opp_avg_blk': ('OppBlk', 'mean'),
            'opp_avg_pf': ('OppPF', 'mean'),
        }

        # Aggregate by Season and Team
        df_mm = df_regular.groupby(['Season', 'TeamID']).agg(**agg_map).reset_index()

        # Compute calculated ratios (Shooting Percentages)
        df_mm['avg_fgr'] = df_mm['avg_fgm'] / df_mm['avg_fga']
        df_mm['avg_fgr3'] = df_mm['avg_fgm3'] / df_mm['avg_fga3']
        df_mm['avg_ftr'] = df_mm['avg_ftm'] / df_mm['avg_fta']
        
        df_mm['opp_avg_fgr'] = df_mm['opp_avg_fgm'] / df_mm['opp_avg_fga']
        df_mm['opp_avg_fgr3'] = df_mm['opp_avg_fgm3'] / df_mm['opp_avg_fga3']
        df_mm['opp_avg_ftr'] = df_mm['opp_avg_ftm'] / df_mm['opp_avg_fta']

        return df_mm

    def _build_features(self, df_scorecards, df_feature_base):
        """
        Joins engineered features onto scorecards.
        """
        # USE THE SINGLE COLUMN NAMES HERE
        feat_cols = self.X_sing_eng_cols 

        df_regular = self._compute_regular_season_features(df_feature_base)
        df_mm_features = self._compute_season_summary_features(df_feature_base)

        # Slice using non-prefixed names
        df_regular_features = df_regular[['GameID', 'PrevGameID', 'Season', 'TeamID'] + feat_cols].copy()

        # ── Regular season games ───────────────────────────────────────────────
        df_scorecard_reg = df_scorecards[df_scorecards['IsRegularSeason'] == True].copy()

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T1_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T2_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        # ── March Madness games ────────────────────────────────────────────────
        df_scorecard_mm = df_scorecards[df_scorecards['IsRegularSeason'] == False].copy()

        # FIX: Slice df_mm_features using feat_cols (non-prefixed)
        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        keep_cols = ['GameID', 'Season', 'DayNum', self.y_col] + self.X_cols
        return pd.concat([df_scorecard_reg, df_scorecard_mm], ignore_index=True)[keep_cols]
    # ── Public interface ───────────────────────────────────────────────────────

    def get_test_data(self):
        X_test = self.df_test[self.X_cols]
        y_test = self.df_test[self.y_col]
        return X_test, y_test

    def get_model(self):
        return self.model
    
    def data_prep(self):
        """Builds features from the instance's scorecards and feature base, populates df_baseline and df_train."""
        self.df_baseline = self._build_features(self.df_scorecards, self.df_feature_base)
        self.df_train = self.df_baseline[self.df_baseline.notna().all(axis=1)]
        print('Data prepped.')

    def split_test_from_train(self):
        """
        Carves the most recent 500 March Madness games out of df_baseline
        into df_test, and removes them from df_train.
        Use this in Phase 1 when you don't have a separate test set.
        """
        df_test = (
            self.df_baseline[self.df_baseline['IsRegularSeason'] == False]
            .sort_values('Season', ascending=False)
            .iloc[0:500]
        )
        test_games = set(df_test['GameID'])
        self.df_test = df_test
        self.df_train = self.df_train[~self.df_train['GameID'].isin(test_games)]
        print(f'Test set carved: {len(df_test)} games.')

    def get_final_test_data(self, df_final_test_scorecards, df_final_test_feature_base):
        """
        Builds features from an external test set and stores in df_test.
        Features are sourced from self.df_feature_base (regular season data),
        since df_final_test_feature_base only contains March Madness games.
        Use this in Phase 2 before evaluate().
        """
        self.df_test = self._build_features(df_final_test_scorecards, self.df_feature_base)
        print('Final test data prepped.')

    def train(self, use_full_data=False):
        """
        use_full_data=False  → train on df_train only (Phases 1 & 2)
        use_full_data=True   → train on all of df_baseline (Phase 3)
        """
        source = self.df_baseline if use_full_data else self.df_train

        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]

        self.model = LogisticRegression(max_iter=2000)
        self.model.fit(X, y)
        print(f'Model trained on {"full dataset" if use_full_data else "training set"}.')

    def evaluate(self):
        """Evaluates the model against df_test."""
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')
        if self.df_test is None or self.df_test.empty:
            raise RuntimeError('No test set available. Call split_test_from_train() or get_final_test_data() first.')

        X_test = self.df_test[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y_test = self.df_test[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X_test.notna().all(axis=1) & y_test.notna()
        X_test, y_test = X_test[mask], y_test[mask]

        df_test_eval = self.df_test[mask].copy()
        df_test_eval['predicted_prob'] = self.model.predict_proba(X_test)[:, 1]

        self.brier = brier_score_loss(y_test, df_test_eval['predicted_prob'])
        print(f'Brier Score: {self.brier:.6f}')
        return self.brier

    def predict(self):
        """
        Generates March Madness win probability predictions for all hypothetical matchups.
        T1 is always the lower TeamID. Returns a dataframe with columns [ID, Pred].

        ID format: '2026_{lower_team_id}_{higher_team_id}'
        Pred: probability that the lower TeamID team wins.
        """
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')

        # Get all unique teams per gender from the March Madness scorecards
        df_mm = self.df_scorecards[self.df_scorecards['IsRegularSeason'] == False].copy()
        season = df_mm['Season'].max()  # predict for the most recent/current season

        # Build all hypothetical matchups within each gender
        records = []
        for is_male in [True, False]:
            teams = sorted(
                df_mm[df_mm['IsMale'] == is_male]['T1_TeamID'].tolist() +
                df_mm[df_mm['IsMale'] == is_male]['T2_TeamID'].tolist()
            )
            teams = sorted(set(teams))

            for i, t1 in enumerate(teams):
                for t2 in teams[i + 1:]:
                    records.append({
                        'IsMale': int(is_male),
                        'IsRegularSeason': 0,
                        'Season': season,
                        'T1_TeamID': t1,
                        'T2_TeamID': t2,
                    })

        df_matchups = pd.DataFrame(records)

        # Join season summary features (same logic as March Madness in _build_features)
        df_mm_features = self._compute_season_summary_features(self.df_feature_base)
        feat_cols = self.X_sing_eng_cols

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        X = df_matchups[self.X_cols].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1)
        X = X[mask]
        df_matchups = df_matchups[mask].copy()

        df_matchups['Pred'] = self.model.predict_proba(X)[:, 1]
        df_matchups['ID'] = df_matchups.apply(
            lambda r: f"{int(r['Season'])}_{int(r['T1_TeamID'])}_{int(r['T2_TeamID'])}", axis=1
        )

        return df_matchups[['ID', 'Pred']].reset_index(drop=True)


### ── Phase workflows ────────────────────────────────────────────────────────────

### Phase 1 — Build and evaluate against a carve-out of training data
inc_logistic = InclusiveLogisticModel(df_exp_scorecards, df_exp_feature_base)
inc_logistic.data_prep()
inc_logistic.split_test_from_train()
inc_logistic.train()
inc_logistic.evaluate()
inc_log_test_X, inc_log_test_y = inc_logistic.get_test_data()
inc_log_model = inc_logistic.get_model()

### Phase 2 — Evaluate against the official final test set
# baseline = BaselineModel(df_exp_scorecards, df_exp_feature_base)
# baseline.data_prep()
# baseline.get_final_test_data(df_final_test_scorecards, df_final_test_feature_base)
# baseline.train()
# baseline.evaluate()

### Phase 3 — Retrain on all data and generate predictions
# baseline = BaselineModel(df_all_scorecard, df_all_features_base)
# baseline.data_prep()
# baseline.train(use_full_data=True)
# baseline_prediction = baseline.predict()

Data prepped.
Test set carved: 500 games.
Model trained on training set.
Brier Score: 0.201038


/opt/homebrew/Cellar/jupyterlab/4.4.4/libexec/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Permutation Importance

In [256]:
cols = inc_log_test_X.columns

pred_prob_og = inc_log_model.predict_proba(inc_log_test_X)
brier_og = brier_score_loss(inc_log_test_y, pred_prob_og)

print(f'Brier OG: {brier_og:.6f}')

brier_list = []
delta_list = []

for col in cols:
    inc_log_test_X_shuffle = inc_log_test_X.copy()
    inc_log_test_X_shuffle[col] = inc_log_test_X[col].sample(frac=1).values
    
    pred_prob = inc_log_model.predict_proba(inc_log_test_X_shuffle)[:, 1]
    brier = brier_score_loss(inc_log_test_y, pred_prob)
    delta = brier_og - brier
    
    brier_list.append(brier)
    delta_list.append(delta)

brier_scores = pd.DataFrame({
    'feature': cols,
    'brier_score': brier_list,
    'delta':delta_list})
brier_scores.sort_values('brier_score')

Brier OG: 0.201038


,feature,brier_score,delta
26,T1_opp_avg_to,0.199369,0.001668
33,T2_avg_fgm3,0.200370,0.000668
53,T2_opp_avg_ast,0.200455,0.000583
15,T1_avg_pf,0.200582,0.000456
23,T1_opp_avg_or,0.200628,0.000410
45,T2_opp_avg_fgm,0.200651,0.000387
41,T2_avg_stl,0.200684,0.000354
35,T2_avg_ftm,0.200809,0.000229
57,T2_opp_avg_pf,0.200827,0.000211
37,T2_avg_or,0.200930,0.000108


## Inclusive XGB

In [260]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss


class InclusiveXGBModel:
    def __init__(self, df_scorecards, df_feature_base):
        self.df_scorecards = df_scorecards
        self.df_feature_base = df_feature_base
        self.X_sing_eng_cols = [
            'avg_scr','avg_fgm','avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr',
            'avg_or','avg_dr', 'avg_ast', 'avg_to', 'avg_stl', 'avg_blk', 'avg_pf',
            'opp_avg_scr','opp_avg_fgm','opp_avg_fgr', 'opp_avg_fgm3', 'opp_avg_fgr3', 'opp_avg_ftm', 'opp_avg_ftr',
            'opp_avg_or','opp_avg_dr', 'opp_avg_ast', 'opp_avg_to', 'opp_avg_stl', 'opp_avg_blk', 'opp_avg_pf']
        self.X_eng_cols = [
            'T1_avg_scr','T1_avg_fgm','T1_avg_fgr', 'T1_avg_fgm3', 'T1_avg_fgr3', 'T1_avg_ftm', 'T1_avg_ftr',
            'T1_avg_or','T1_avg_dr', 'T1_avg_ast', 'T1_avg_to', 'T1_avg_stl', 'T1_avg_blk', 'T1_avg_pf',
            'T1_opp_avg_scr','T1_opp_avg_fgm','T1_opp_avg_fgr', 'T1_opp_avg_fgm3', 'T1_opp_avg_fgr3', 'T1_opp_avg_ftm', 'T1_opp_avg_ftr',
            'T1_opp_avg_or','T1_opp_avg_dr', 'T1_opp_avg_ast', 'T1_opp_avg_to', 'T1_opp_avg_stl', 'T1_opp_avg_blk', 'T1_opp_avg_pf',
            
            'T2_avg_scr','T2_avg_fgm', 'T2_avg_fgr', 'T2_avg_fgm3', 'T2_avg_fgr3', 'T2_avg_ftm', 'T2_avg_ftr',
            'T2_avg_or','T2_avg_dr', 'T2_avg_ast', 'T2_avg_to', 'T2_avg_stl', 'T2_avg_blk', 'T2_avg_pf',
            'T2_opp_avg_scr','T2_opp_avg_fgm', 'T2_opp_avg_fgr', 'T2_opp_avg_fgm3', 'T2_opp_avg_fgr3', 'T2_opp_avg_ftm', 'T2_opp_avg_ftr',
            'T2_opp_avg_or','T2_opp_avg_dr', 'T2_opp_avg_ast', 'T2_opp_avg_to', 'T2_opp_avg_stl', 'T2_opp_avg_blk', 'T2_opp_avg_pf',
        ]
        self.X_cols = ['IsMale', 'IsRegularSeason'] + self.X_eng_cols
        self.y_col = 'T1_win'
        self.model = None
        self.df_train = None
        self.df_test = None
        self.df_baseline = None

    # ── Private helpers ────────────────────────────────────────────────────────

    def _compute_regular_season_features(self, df_feature_base):
        """Rolling per-game averages for regular season games."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()
        df_regular = df_regular.sort_values(['Season', 'TeamID', 'DayNum'])

        for col, new_col in [
            ('Score', 'avg_scr'), ('FGM', 'avg_fgm'), ('FGA', 'avg_fga'), ('FGM3', 'avg_fgm3'), 
            ('FGA3', 'avg_fga3'), ('FTM', 'avg_ftm'), ('FTA', 'avg_fta'),
            ('OR', 'avg_or'), ('DR', 'avg_dr'), ('Ast', 'avg_ast'), 
            ('TO', 'avg_to'), ('Stl', 'avg_stl'), ('Blk', 'avg_blk'), ('PF', 'avg_pf'),

            ('OppScore', 'opp_avg_scr'), ('OppFGM', 'opp_avg_fgm'), ('OppFGA', 'opp_avg_fga'), ('OppFGM3', 'opp_avg_fgm3'), 
            ('OppFGA3', 'opp_avg_fga3'), ('OppFTM', 'opp_avg_ftm'), ('OppFTA', 'opp_avg_fta'),
            ('OppOR', 'opp_avg_or'), ('OppDR', 'opp_avg_dr'), ('OppAst', 'opp_avg_ast'), 
            ('OppTO', 'opp_avg_to'), ('OppStl', 'opp_avg_stl'), ('OppBlk', 'opp_avg_blk'), ('OppPF', 'opp_avg_pf'),
        ]:
            df_regular[new_col] = (
                df_regular
                .groupby(['Season', 'TeamID'])[col]
                .expanding()
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

        df_regular['PrevGameID'] = (
            df_regular
            .sort_values(['TeamID', 'Season', 'DayNum'])
            .groupby(['TeamID', 'Season'])['GameID']
            .shift(-1)
        )

        df_regular['avg_fgr'] = df_regular['avg_fgm'] / df_regular['avg_fga']
        df_regular['avg_fgr3'] = df_regular['avg_fgm3'] / df_regular['avg_fga3']
        df_regular['avg_ftr'] = df_regular['avg_ftm'] / df_regular['avg_fta']
        df_regular['opp_avg_fgr'] = df_regular['opp_avg_fgm'] / df_regular['opp_avg_fga']
        df_regular['opp_avg_fgr3'] = df_regular['opp_avg_fgm3'] / df_regular['opp_avg_fga3']
        df_regular['opp_avg_ftr'] = df_regular['opp_avg_ftm'] / df_regular['opp_avg_fta']

        return df_regular

    def _compute_season_summary_features(self, df_feature_base):
        """Full-season averages per team, used for March Madness matchups."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()

        # Define the mapping for all base stats to be averaged
        agg_map = {
            # Team Offense
            'avg_scr': ('Score', 'mean'),
            'avg_fgm': ('FGM', 'mean'),
            'avg_fga': ('FGA', 'mean'),
            'avg_fgm3': ('FGM3', 'mean'),
            'avg_fga3': ('FGA3', 'mean'),
            'avg_ftm': ('FTM', 'mean'),
            'avg_fta': ('FTA', 'mean'),
            'avg_or': ('OR', 'mean'),
            'avg_dr': ('DR', 'mean'),
            'avg_ast': ('Ast', 'mean'),
            'avg_to': ('TO', 'mean'),
            'avg_stl': ('Stl', 'mean'),
            'avg_blk': ('Blk', 'mean'),
            'avg_pf': ('PF', 'mean'),
            
            # Team Defense
            'opp_avg_scr': ('OppScore', 'mean'),
            'opp_avg_fgm': ('OppFGM', 'mean'),
            'opp_avg_fga': ('OppFGA', 'mean'),
            'opp_avg_fgm3': ('OppFGM3', 'mean'),
            'opp_avg_fga3': ('OppFGA3', 'mean'),
            'opp_avg_ftm': ('OppFTM', 'mean'),
            'opp_avg_fta': ('OppFTA', 'mean'),
            'opp_avg_or': ('OppOR', 'mean'),
            'opp_avg_dr': ('OppDR', 'mean'),
            'opp_avg_ast': ('OppAst', 'mean'),
            'opp_avg_to': ('OppTO', 'mean'),
            'opp_avg_stl': ('OppStl', 'mean'),
            'opp_avg_blk': ('OppBlk', 'mean'),
            'opp_avg_pf': ('OppPF', 'mean'),
        }

        # Aggregate by Season and Team
        df_mm = df_regular.groupby(['Season', 'TeamID']).agg(**agg_map).reset_index()

        # Compute calculated ratios (Shooting Percentages)
        df_mm['avg_fgr'] = df_mm['avg_fgm'] / df_mm['avg_fga']
        df_mm['avg_fgr3'] = df_mm['avg_fgm3'] / df_mm['avg_fga3']
        df_mm['avg_ftr'] = df_mm['avg_ftm'] / df_mm['avg_fta']
        
        df_mm['opp_avg_fgr'] = df_mm['opp_avg_fgm'] / df_mm['opp_avg_fga']
        df_mm['opp_avg_fgr3'] = df_mm['opp_avg_fgm3'] / df_mm['opp_avg_fga3']
        df_mm['opp_avg_ftr'] = df_mm['opp_avg_ftm'] / df_mm['opp_avg_fta']

        return df_mm

    def _build_features(self, df_scorecards, df_feature_base):
        """
        Joins engineered features onto scorecards.
        """
        # USE THE SINGLE COLUMN NAMES HERE
        feat_cols = self.X_sing_eng_cols 

        df_regular = self._compute_regular_season_features(df_feature_base)
        df_mm_features = self._compute_season_summary_features(df_feature_base)

        # Slice using non-prefixed names
        df_regular_features = df_regular[['GameID', 'PrevGameID', 'Season', 'TeamID'] + feat_cols].copy()

        # ── Regular season games ───────────────────────────────────────────────
        df_scorecard_reg = df_scorecards[df_scorecards['IsRegularSeason'] == True].copy()

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T1_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T2_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        # ── March Madness games ────────────────────────────────────────────────
        df_scorecard_mm = df_scorecards[df_scorecards['IsRegularSeason'] == False].copy()

        # FIX: Slice df_mm_features using feat_cols (non-prefixed)
        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        keep_cols = ['GameID', 'Season', 'DayNum', self.y_col] + self.X_cols
        return pd.concat([df_scorecard_reg, df_scorecard_mm], ignore_index=True)[keep_cols]
    # ── Public interface ───────────────────────────────────────────────────────

    def _kfold_brier(self, X, y, strat_col, params, n_splits=5):
        """
        Runs stratified K-fold CV and returns mean train and val Brier scores.
        strat_col is a 1-D array used for stratification (IsRegularSeason).
        """
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        train_scores, val_scores = [], []

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, strat_col)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model = XGBClassifier(**params, random_state=42, eval_metric='logloss', verbosity=0)

            # Fit with early stopping using the val fold as the eval set.
            # XGB's internal early stopping watches logloss; we then compute
            # Brier ourselves so the reported metric is consistent.
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_tr, y_tr), (X_val, y_val)],
                verbose=False,
            )

            train_pred = model.predict_proba(X_tr)[:, 1]
            val_pred   = model.predict_proba(X_val)[:, 1]

            train_scores.append(brier_score_loss(y_tr,  train_pred))
            val_scores.append(brier_score_loss(y_val, val_pred))

        return np.mean(train_scores), np.mean(val_scores)

    def get_test_data(self):
        X_test = self.df_test[self.X_cols]
        y_test = self.df_test[self.y_col]
        return X_test, y_test

    def get_model(self):
        return self.model
    
    def data_prep(self):
        """Builds features from the instance's scorecards and feature base, populates df_baseline and df_train."""
        self.df_baseline = self._build_features(self.df_scorecards, self.df_feature_base)
        self.df_train = self.df_baseline[self.df_baseline.notna().all(axis=1)]
        print('Data prepped.')

    def split_test_from_train(self):
        """
        Carves the most recent 500 March Madness games out of df_baseline
        into df_test, and removes them from df_train.
        Use this in Phase 1 when you don't have a separate test set.
        """
        df_test = (
            self.df_baseline[self.df_baseline['IsRegularSeason'] == False]
            .sort_values('Season', ascending=False)
            .iloc[0:500]
        )
        test_games = set(df_test['GameID'])
        self.df_test = df_test
        self.df_train = self.df_train[~self.df_train['GameID'].isin(test_games)]
        print(f'Test set carved: {len(df_test)} games.')

    def get_final_test_data(self, df_final_test_scorecards, df_final_test_feature_base):
        """
        Builds features from an external test set and stores in df_test.
        Features are sourced from self.df_feature_base (regular season data),
        since df_final_test_feature_base only contains March Madness games.
        Use this in Phase 2 before evaluate().
        """
        self.df_test = self._build_features(df_final_test_scorecards, self.df_feature_base)
        print('Final test data prepped.')

    def hyper_param_discovery_train(self):
        """
        Searches over a grid of XGBoost hyperparameters using 5-fold CV
        stratified by IsRegularSeason. Optimises for Brier score.

        Overfitting guardrail: if the val Brier is *better* (lower) than the
        train Brier for 3 consecutive candidates, the search halts early —
        this indicates the model is underfitting and further relaxing
        regularisation won't help on unseen data.

        Returns the best param dict (lowest mean val Brier).
        """
        source = self.df_train.copy()
        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]
        strat_col = source[mask]['IsRegularSeason'].astype(int)

        # ── Hyperparameter grid ────────────────────────────────────────────────
        # Kept intentionally conservative to favour generalisation.
        param_grid = [
            {'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
             'subsample': ss, 'colsample_bytree': cs, 'reg_lambda': lam}
            for n   in [200, 400, 600]
            for d   in [3, 4, 5]
            for lr  in [0.05, 0.01]
            for ss  in [0.7, 0.85]
            for cs  in [0.7, 0.85]
            for lam in [1.0, 2.0]
        ]

        best_val_brier  = float('inf')
        best_params     = None
        results         = []

        # Overfitting guardrail state:
        # counts consecutive rounds where val_brier < train_brier
        overfit_guard_count = 0
        OVERFIT_PATIENCE    = 3

        print(f'Hyperparameter search: {len(param_grid)} candidates, 5-fold CV each.')

        for i, params in enumerate(param_grid):
            train_brier, val_brier = self._kfold_brier(X, y, strat_col, params)
            results.append({**params, 'train_brier': train_brier, 'val_brier': val_brier})

            gap = val_brier - train_brier  # positive = overfitting, negative = underfitting

            status = ''
            if val_brier < best_val_brier:
                best_val_brier = val_brier
                best_params    = params
                status = ' ← best so far'

            print(
                f'  [{i+1:>4}/{len(param_grid)}] val={val_brier:.6f}  '
                f'train={train_brier:.6f}  gap={gap:+.6f}{status}'
            )

            # Guardrail: val better than train signals the model is
            # underfitting (or the split is noisy). Three in a row → stop.
            if val_brier < train_brier:
                overfit_guard_count += 1
                if overfit_guard_count >= OVERFIT_PATIENCE:
                    print(
                        f'\n  [Guardrail] Val Brier < Train Brier for '
                        f'{OVERFIT_PATIENCE} consecutive candidates. '
                        f'Stopping search early to avoid underfitting regime.'
                    )
                    break
            else:
                overfit_guard_count = 0  # reset on any candidate where val ≥ train

        self.hyper_param_results_ = pd.DataFrame(results).sort_values('val_brier')
        print(f'\nBest params (val Brier={best_val_brier:.6f}):\n  {best_params}')
        return best_params

    def train(self, hyper_params=None, use_full_data=False):
        """
        Trains an XGBoost model with the supplied hyper_params.
        Falls back to sensible defaults if hyper_params is None.

        Overfitting guardrail (per-tree early stopping):
          Uses a 10 % internal hold-out as the watch set. If val Brier
          improves over train Brier for 3 consecutive boosting rounds,
          training stops — identical logic to hyper_param_discovery_train.

        use_full_data=False  → train on df_train only (Phases 1 & 2)
        use_full_data=True   → train on all of df_baseline (Phase 3)
        """
        source = self.df_baseline if use_full_data else self.df_train

        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]

        if hyper_params is None:
            hyper_params = {
                'n_estimators': 400,
                'max_depth': 4,
                'learning_rate': 0.05,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_lambda': 1.0,
            }

        # ── Internal watch-set for per-tree guardrail ──────────────────────────
        # Stratify the 10 % hold-out by IsRegularSeason so both game types
        # are represented in the watch set.
        strat_col = source[mask]['IsRegularSeason'].astype(int)
        skf_single = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
        train_idx, watch_idx = next(skf_single.split(X, strat_col))
        X_tr,    X_watch  = X.iloc[train_idx],    X.iloc[watch_idx]
        y_tr,    y_watch  = y.iloc[train_idx],    y.iloc[watch_idx]

        # ── Custom callback: stop when val Brier < train Brier 3× in a row ────
        from xgboost.callback import TrainingCallback

        class BrierOverfitGuard(TrainingCallback):
            """
            Stops boosting if val Brier < train Brier for `patience`
            consecutive rounds (underfitting / noise regime).
            Also stops if val Brier has not improved for `early_stop` rounds
            (classic early stopping on Brier rather than logloss).
            """
            def __init__(self, X_tr, y_tr, X_val, y_val,
                         patience=3, early_stop=30, check_every=10):
                self.X_tr, self.y_tr   = X_tr, y_tr
                self.X_val, self.y_val = X_val, y_val
                self.patience          = patience
                self.early_stop        = early_stop
                self.check_every       = check_every
                self._under_count      = 0   # consecutive rounds val < train
                self._best_val         = float('inf')
                self._rounds_no_improv = 0

            def after_iteration(self, model, epoch, evals_log):
                if (epoch + 1) % self.check_every != 0:
                    return False  # don't stop

                # Booster.inplace_predict accepts numpy arrays directly
                train_prob = model.inplace_predict(self.X_tr.values)
                val_prob   = model.inplace_predict(self.X_val.values)

                train_b = brier_score_loss(self.y_tr,  train_prob)
                val_b   = brier_score_loss(self.y_val, val_prob)
                gap     = val_b - train_b

                # Classic early stopping: val Brier not improving
                if val_b < self._best_val:
                    self._best_val         = val_b
                    self._rounds_no_improv = 0
                else:
                    self._rounds_no_improv += 1
                    if self._rounds_no_improv >= self.early_stop // self.check_every:
                        print(
                            f'  [EarlyStopping] Val Brier stalled for '
                            f'{self._rounds_no_improv * self.check_every} rounds '
                            f'(val={val_b:.6f}). Stopping.'
                        )
                        return True  # stop

                # Guardrail: underfitting regime
                if val_b < train_b:
                    self._under_count += 1
                    if self._under_count >= self.patience:
                        print(
                            f'  [Guardrail] Val Brier < Train Brier for '
                            f'{self.patience} consecutive checks '
                            f'(gap={gap:+.6f}). Stopping.'
                        )
                        return True
                else:
                    self._under_count = 0

                return False  # continue training


        guard = BrierOverfitGuard(X_tr, y_tr, X_watch, y_watch,
                                  patience=3, early_stop=30, check_every=10)

        self.model = XGBClassifier(
            **hyper_params,
            random_state=42,
            eval_metric='logloss',
            verbosity=0,
            callbacks=[guard],
        )
        self.model.fit(
            X_tr, y_tr,
            eval_set=[(X_tr, y_tr), (X_watch, y_watch)],
            verbose=False,
        )
        print(f'Model trained on {"full dataset" if use_full_data else "training set"}.')

    def evaluate(self):
        """Evaluates the model against df_test."""
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')
        if self.df_test is None or self.df_test.empty:
            raise RuntimeError('No test set available. Call split_test_from_train() or get_final_test_data() first.')

        X_test = self.df_test[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y_test = self.df_test[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X_test.notna().all(axis=1) & y_test.notna()
        X_test, y_test = X_test[mask], y_test[mask]

        df_test_eval = self.df_test[mask].copy()
        df_test_eval['predicted_prob'] = self.model.predict_proba(X_test)[:, 1]

        self.brier = brier_score_loss(y_test, df_test_eval['predicted_prob'])
        print(f'Brier Score: {self.brier:.6f}')
        return self.brier

    def predict(self):
        """
        Generates March Madness win probability predictions for all hypothetical matchups.
        T1 is always the lower TeamID. Returns a dataframe with columns [ID, Pred].

        ID format: '2026_{lower_team_id}_{higher_team_id}'
        Pred: probability that the lower TeamID team wins.
        """
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')

        # Get all unique teams per gender from the March Madness scorecards
        df_mm = self.df_scorecards[self.df_scorecards['IsRegularSeason'] == False].copy()
        season = df_mm['Season'].max()  # predict for the most recent/current season

        # Build all hypothetical matchups within each gender
        records = []
        for is_male in [True, False]:
            teams = sorted(
                df_mm[df_mm['IsMale'] == is_male]['T1_TeamID'].tolist() +
                df_mm[df_mm['IsMale'] == is_male]['T2_TeamID'].tolist()
            )
            teams = sorted(set(teams))

            for i, t1 in enumerate(teams):
                for t2 in teams[i + 1:]:
                    records.append({
                        'IsMale': int(is_male),
                        'IsRegularSeason': 0,
                        'Season': season,
                        'T1_TeamID': t1,
                        'T2_TeamID': t2,
                    })

        df_matchups = pd.DataFrame(records)

        # Join season summary features (same logic as March Madness in _build_features)
        df_mm_features = self._compute_season_summary_features(self.df_feature_base)
        feat_cols = self.X_sing_eng_cols

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        X = df_matchups[self.X_cols].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1)
        X = X[mask]
        df_matchups = df_matchups[mask].copy()

        df_matchups['Pred'] = self.model.predict_proba(X)[:, 1]
        df_matchups['ID'] = df_matchups.apply(
            lambda r: f"{int(r['Season'])}_{int(r['T1_TeamID'])}_{int(r['T2_TeamID'])}", axis=1
        )

        return df_matchups[['ID', 'Pred']].reset_index(drop=True)


### ── Phase workflows ────────────────────────────────────────────────────────────

### Phase 1 — Evaluate hyperparameters and overfitting risk
inc_xgb = InclusiveXGBModel(df_exp_scorecards, df_exp_feature_base)
inc_xgb.data_prep()
inc_xgb.split_test_from_train()
xgb_hyper_params = inc_xgb.hyper_param_discovery_train()   # grid search → best params
inc_xgb.train(xgb_hyper_params)                            # final train with guardrail callback
inc_xgb.evaluate()
inc_xgb_test_X, inc_xgb_test_y = inc_xgb.get_test_data()
inc_xgb_model = inc_xgb.get_model()
# Optional: inspect CV results table
# inc_xgb.hyper_param_results_


### Phase 2 — Evaluate against the official final test set
# inc_xgb = InclusiveXGBModel(df_exp_scorecards, df_exp_feature_base)
# inc_xgb.data_prep()
# inc_xgb.get_final_test_data(df_final_test_scorecards, df_final_test_feature_base)
# inc_xgb.train(xgb_hyper_params)
# inc_xgb.evaluate()

### Phase 3 — Retrain on all data and generate predictions
# inc_xgb = InclusiveXGBModel(df_all_scorecard, df_all_features_base)
# inc_xgb.data_prep()
# inc_xgb.train(xgb_hyper_params, use_full_data=True)
# inc_xgb_prediction = inc_xgb.predict()

Data prepped.
Test set carved: 500 games.
  [EarlyStopping] Val Brier stalled for 30 rounds (val=0.192196). Stopping.
Model trained on training set.
Brier Score: 0.199285


In [261]:
cols = inc_xgb_test_X.columns

pred_prob_og = inc_xgb_model.predict_proba(inc_xgb_test_X)
brier_og = brier_score_loss(inc_xgb_test_y, pred_prob_og)

print(f'Brier OG: {brier_og:.6f}')

brier_list = []
delta_list = []

for col in cols:
    inc_xgb_test_X_shuffle = inc_xgb_test_X.copy()
    inc_xgb_test_X_shuffle[col] = inc_xgb_test_X[col].sample(frac=1).values
    
    pred_prob = inc_xgb_model.predict_proba(inc_xgb_test_X_shuffle)[:, 1]
    brier = brier_score_loss(inc_xgb_test_y, pred_prob)
    delta = brier_og - brier
    
    brier_list.append(brier)
    delta_list.append(delta)

brier_scores = pd.DataFrame({
    'feature': cols,
    'brier_score': brier_list,
    'delta':delta_list})
brier_scores.sort_values('brier_score')

T1_avg_ast
T1_opp_avg_ast
T2_avg_ast
T2_opp_avg_ast

T2_opp_avg_or
T2_avg_or
T1_avg_or
T1_opp_avg_or

Brier OG: 0.199285


,feature,brier_score,delta
11,T1_avg_ast,0.197665,0.001620
51,T2_opp_avg_or,0.197901,0.001384
37,T2_avg_or,0.198286,0.000998
20,T1_opp_avg_fgr3,0.198659,0.000625
29,T1_opp_avg_pf,0.198697,0.000588
9,T1_avg_or,0.198719,0.000566
25,T1_opp_avg_ast,0.198756,0.000529
7,T1_avg_ftm,0.198788,0.000496
39,T2_avg_ast,0.198800,0.000485
24,T1_opp_avg_dr,0.198803,0.000481


## XGBoost Inclusive - Filtered Features

In [265]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss


class InclusiveXGBModelFiltered:
    def __init__(self, df_scorecards, df_feature_base):
        self.df_scorecards = df_scorecards
        self.df_feature_base = df_feature_base
        self.X_sing_eng_cols = [
            'avg_scr','avg_fgm','avg_fgr', 'avg_fgm3', 'avg_fgr3', 'avg_ftm', 'avg_ftr',
            'avg_dr', 'avg_to', 'avg_stl', 'avg_blk', 'avg_pf',
            'opp_avg_scr','opp_avg_fgm','opp_avg_fgr', 'opp_avg_fgm3', 'opp_avg_fgr3', 'opp_avg_ftm', 'opp_avg_ftr',
            'opp_avg_dr', 'opp_avg_to', 'opp_avg_stl', 'opp_avg_blk', 'opp_avg_pf']
        self.X_eng_cols = [
            'T1_avg_scr','T1_avg_fgm','T1_avg_fgr', 'T1_avg_fgm3', 'T1_avg_fgr3', 'T1_avg_ftm', 'T1_avg_ftr',
            'T1_avg_dr', 'T1_avg_to', 'T1_avg_stl', 'T1_avg_blk', 'T1_avg_pf',
            'T1_opp_avg_scr','T1_opp_avg_fgm','T1_opp_avg_fgr', 'T1_opp_avg_fgm3', 'T1_opp_avg_fgr3', 'T1_opp_avg_ftm', 'T1_opp_avg_ftr',
            'T1_opp_avg_dr', 'T1_opp_avg_to', 'T1_opp_avg_stl', 'T1_opp_avg_blk', 'T1_opp_avg_pf',
            
            'T2_avg_scr','T2_avg_fgm', 'T2_avg_fgr', 'T2_avg_fgm3', 'T2_avg_fgr3', 'T2_avg_ftm', 'T2_avg_ftr',
            'T2_avg_dr', 'T2_avg_to', 'T2_avg_stl', 'T2_avg_blk', 'T2_avg_pf',
            'T2_opp_avg_scr','T2_opp_avg_fgm', 'T2_opp_avg_fgr', 'T2_opp_avg_fgm3', 'T2_opp_avg_fgr3', 'T2_opp_avg_ftm', 'T2_opp_avg_ftr',
            'T2_opp_avg_dr', 'T2_opp_avg_to', 'T2_opp_avg_stl', 'T2_opp_avg_blk', 'T2_opp_avg_pf',
        ]
        self.X_cols = ['IsMale', 'IsRegularSeason'] + self.X_eng_cols
        self.y_col = 'T1_win'
        self.model = None
        self.df_train = None
        self.df_test = None
        self.df_baseline = None

    # ── Private helpers ────────────────────────────────────────────────────────

    def _compute_regular_season_features(self, df_feature_base):
        """Rolling per-game averages for regular season games."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()
        df_regular = df_regular.sort_values(['Season', 'TeamID', 'DayNum'])

        for col, new_col in [
            ('Score', 'avg_scr'), ('FGM', 'avg_fgm'), ('FGA', 'avg_fga'), ('FGM3', 'avg_fgm3'), 
            ('FGA3', 'avg_fga3'), ('FTM', 'avg_ftm'), ('FTA', 'avg_fta'),
            ('DR', 'avg_dr'), 
            ('TO', 'avg_to'), ('Stl', 'avg_stl'), ('Blk', 'avg_blk'), ('PF', 'avg_pf'),

            ('OppScore', 'opp_avg_scr'), ('OppFGM', 'opp_avg_fgm'), ('OppFGA', 'opp_avg_fga'), ('OppFGM3', 'opp_avg_fgm3'), 
            ('OppFGA3', 'opp_avg_fga3'), ('OppFTM', 'opp_avg_ftm'), ('OppFTA', 'opp_avg_fta'),
            ('OppDR', 'opp_avg_dr'), 
            ('OppTO', 'opp_avg_to'), ('OppStl', 'opp_avg_stl'), ('OppBlk', 'opp_avg_blk'), ('OppPF', 'opp_avg_pf'),
        ]:
            df_regular[new_col] = (
                df_regular
                .groupby(['Season', 'TeamID'])[col]
                .expanding()
                .mean()
                .reset_index(level=[0, 1], drop=True)
            )

        df_regular['PrevGameID'] = (
            df_regular
            .sort_values(['TeamID', 'Season', 'DayNum'])
            .groupby(['TeamID', 'Season'])['GameID']
            .shift(-1)
        )

        df_regular['avg_fgr'] = df_regular['avg_fgm'] / df_regular['avg_fga']
        df_regular['avg_fgr3'] = df_regular['avg_fgm3'] / df_regular['avg_fga3']
        df_regular['avg_ftr'] = df_regular['avg_ftm'] / df_regular['avg_fta']
        df_regular['opp_avg_fgr'] = df_regular['opp_avg_fgm'] / df_regular['opp_avg_fga']
        df_regular['opp_avg_fgr3'] = df_regular['opp_avg_fgm3'] / df_regular['opp_avg_fga3']
        df_regular['opp_avg_ftr'] = df_regular['opp_avg_ftm'] / df_regular['opp_avg_fta']

        return df_regular

    def _compute_season_summary_features(self, df_feature_base):
        """Full-season averages per team, used for March Madness matchups."""
        df_regular = df_feature_base[df_feature_base['IsRegularSeason'] == True].copy()

        # Define the mapping for all base stats to be averaged
        agg_map = {
            # Team Offense
            'avg_scr': ('Score', 'mean'),
            'avg_fgm': ('FGM', 'mean'),
            'avg_fga': ('FGA', 'mean'),
            'avg_fgm3': ('FGM3', 'mean'),
            'avg_fga3': ('FGA3', 'mean'),
            'avg_ftm': ('FTM', 'mean'),
            'avg_fta': ('FTA', 'mean'),
            'avg_dr': ('DR', 'mean'),
            'avg_to': ('TO', 'mean'),
            'avg_stl': ('Stl', 'mean'),
            'avg_blk': ('Blk', 'mean'),
            'avg_pf': ('PF', 'mean'),
            
            # Team Defense
            'opp_avg_scr': ('OppScore', 'mean'),
            'opp_avg_fgm': ('OppFGM', 'mean'),
            'opp_avg_fga': ('OppFGA', 'mean'),
            'opp_avg_fgm3': ('OppFGM3', 'mean'),
            'opp_avg_fga3': ('OppFGA3', 'mean'),
            'opp_avg_ftm': ('OppFTM', 'mean'),
            'opp_avg_fta': ('OppFTA', 'mean'),
            'opp_avg_dr': ('OppDR', 'mean'),
            'opp_avg_to': ('OppTO', 'mean'),
            'opp_avg_stl': ('OppStl', 'mean'),
            'opp_avg_blk': ('OppBlk', 'mean'),
            'opp_avg_pf': ('OppPF', 'mean'),
        }

        # Aggregate by Season and Team
        df_mm = df_regular.groupby(['Season', 'TeamID']).agg(**agg_map).reset_index()

        # Compute calculated ratios (Shooting Percentages)
        df_mm['avg_fgr'] = df_mm['avg_fgm'] / df_mm['avg_fga']
        df_mm['avg_fgr3'] = df_mm['avg_fgm3'] / df_mm['avg_fga3']
        df_mm['avg_ftr'] = df_mm['avg_ftm'] / df_mm['avg_fta']
        
        df_mm['opp_avg_fgr'] = df_mm['opp_avg_fgm'] / df_mm['opp_avg_fga']
        df_mm['opp_avg_fgr3'] = df_mm['opp_avg_fgm3'] / df_mm['opp_avg_fga3']
        df_mm['opp_avg_ftr'] = df_mm['opp_avg_ftm'] / df_mm['opp_avg_fta']

        return df_mm

    def _build_features(self, df_scorecards, df_feature_base):
        """
        Joins engineered features onto scorecards.
        """
        # USE THE SINGLE COLUMN NAMES HERE
        feat_cols = self.X_sing_eng_cols 

        df_regular = self._compute_regular_season_features(df_feature_base)
        df_mm_features = self._compute_season_summary_features(df_feature_base)

        # Slice using non-prefixed names
        df_regular_features = df_regular[['GameID', 'PrevGameID', 'Season', 'TeamID'] + feat_cols].copy()

        # ── Regular season games ───────────────────────────────────────────────
        df_scorecard_reg = df_scorecards[df_scorecards['IsRegularSeason'] == True].copy()

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T1_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_reg = df_scorecard_reg.merge(
            df_regular_features[['PrevGameID', 'TeamID'] + feat_cols],
            left_on=['GameID', 'T2_TeamID'],
            right_on=['PrevGameID', 'TeamID'],
            how='left'
        ).drop(columns=['PrevGameID', 'TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        # ── March Madness games ────────────────────────────────────────────────
        df_scorecard_mm = df_scorecards[df_scorecards['IsRegularSeason'] == False].copy()

        # FIX: Slice df_mm_features using feat_cols (non-prefixed)
        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_scorecard_mm = df_scorecard_mm.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        keep_cols = ['GameID', 'Season', 'DayNum', self.y_col] + self.X_cols
        return pd.concat([df_scorecard_reg, df_scorecard_mm], ignore_index=True)[keep_cols]
    # ── Public interface ───────────────────────────────────────────────────────

    def _kfold_brier(self, X, y, strat_col, params, n_splits=5):
        """
        Runs stratified K-fold CV and returns mean train and val Brier scores.
        strat_col is a 1-D array used for stratification (IsRegularSeason).
        """
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        train_scores, val_scores = [], []

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, strat_col)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model = XGBClassifier(**params, random_state=42, eval_metric='logloss', verbosity=0)

            # Fit with early stopping using the val fold as the eval set.
            # XGB's internal early stopping watches logloss; we then compute
            # Brier ourselves so the reported metric is consistent.
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_tr, y_tr), (X_val, y_val)],
                verbose=False,
            )

            train_pred = model.predict_proba(X_tr)[:, 1]
            val_pred   = model.predict_proba(X_val)[:, 1]

            train_scores.append(brier_score_loss(y_tr,  train_pred))
            val_scores.append(brier_score_loss(y_val, val_pred))

        return np.mean(train_scores), np.mean(val_scores)

    def get_test_data(self):
        X_test = self.df_test[self.X_cols]
        y_test = self.df_test[self.y_col]
        return X_test, y_test

    def get_model(self):
        return self.model
    
    def data_prep(self):
        """Builds features from the instance's scorecards and feature base, populates df_baseline and df_train."""
        self.df_baseline = self._build_features(self.df_scorecards, self.df_feature_base)
        self.df_train = self.df_baseline[self.df_baseline.notna().all(axis=1)]
        print('Data prepped.')

    def split_test_from_train(self):
        """
        Carves the most recent 500 March Madness games out of df_baseline
        into df_test, and removes them from df_train.
        Use this in Phase 1 when you don't have a separate test set.
        """
        df_test = (
            self.df_baseline[self.df_baseline['IsRegularSeason'] == False]
            .sort_values('Season', ascending=False)
            .iloc[0:500]
        )
        test_games = set(df_test['GameID'])
        self.df_test = df_test
        self.df_train = self.df_train[~self.df_train['GameID'].isin(test_games)]
        print(f'Test set carved: {len(df_test)} games.')

    def get_final_test_data(self, df_final_test_scorecards, df_final_test_feature_base):
        """
        Builds features from an external test set and stores in df_test.
        Features are sourced from self.df_feature_base (regular season data),
        since df_final_test_feature_base only contains March Madness games.
        Use this in Phase 2 before evaluate().
        """
        self.df_test = self._build_features(df_final_test_scorecards, self.df_feature_base)
        print('Final test data prepped.')

    def hyper_param_discovery_train(self):
        """
        Searches over a grid of XGBoost hyperparameters using 5-fold CV
        stratified by IsRegularSeason. Optimises for Brier score.

        Overfitting guardrail: if the val Brier is *better* (lower) than the
        train Brier for 3 consecutive candidates, the search halts early —
        this indicates the model is underfitting and further relaxing
        regularisation won't help on unseen data.

        Returns the best param dict (lowest mean val Brier).
        """
        source = self.df_train.copy()
        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]
        strat_col = source[mask]['IsRegularSeason'].astype(int)

        # ── Hyperparameter grid ────────────────────────────────────────────────
        # Kept intentionally conservative to favour generalisation.
        param_grid = [
            {'n_estimators': n, 'max_depth': d, 'learning_rate': lr,
             'subsample': ss, 'colsample_bytree': cs, 'reg_lambda': lam}
            for n   in [200, 400, 600]
            for d   in [3, 4, 5]
            for lr  in [0.05, 0.01]
            for ss  in [0.7, 0.85]
            for cs  in [0.7, 0.85]
            for lam in [1.0, 2.0]
        ]

        best_val_brier  = float('inf')
        best_params     = None
        results         = []

        # Overfitting guardrail state:
        # counts consecutive rounds where val_brier < train_brier
        overfit_guard_count = 0
        OVERFIT_PATIENCE    = 3

        print(f'Hyperparameter search: {len(param_grid)} candidates, 5-fold CV each.')

        for i, params in enumerate(param_grid):
            train_brier, val_brier = self._kfold_brier(X, y, strat_col, params)
            results.append({**params, 'train_brier': train_brier, 'val_brier': val_brier})

            gap = val_brier - train_brier  # positive = overfitting, negative = underfitting

            status = ''
            if val_brier < best_val_brier:
                best_val_brier = val_brier
                best_params    = params
                status = ' ← best so far'

            print(
                f'  [{i+1:>4}/{len(param_grid)}] val={val_brier:.6f}  '
                f'train={train_brier:.6f}  gap={gap:+.6f}{status}'
            )

            # Guardrail: val better than train signals the model is
            # underfitting (or the split is noisy). Three in a row → stop.
            if val_brier < train_brier:
                overfit_guard_count += 1
                if overfit_guard_count >= OVERFIT_PATIENCE:
                    print(
                        f'\n  [Guardrail] Val Brier < Train Brier for '
                        f'{OVERFIT_PATIENCE} consecutive candidates. '
                        f'Stopping search early to avoid underfitting regime.'
                    )
                    break
            else:
                overfit_guard_count = 0  # reset on any candidate where val ≥ train

        self.hyper_param_results_ = pd.DataFrame(results).sort_values('val_brier')
        print(f'\nBest params (val Brier={best_val_brier:.6f}):\n  {best_params}')
        return best_params

    def train(self, hyper_params=None, use_full_data=False):
        """
        Trains an XGBoost model with the supplied hyper_params.
        Falls back to sensible defaults if hyper_params is None.

        Overfitting guardrail (per-tree early stopping):
          Uses a 10 % internal hold-out as the watch set. If val Brier
          improves over train Brier for 3 consecutive boosting rounds,
          training stops — identical logic to hyper_param_discovery_train.

        use_full_data=False  → train on df_train only (Phases 1 & 2)
        use_full_data=True   → train on all of df_baseline (Phase 3)
        """
        source = self.df_baseline if use_full_data else self.df_train

        X = source[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y = source[self.y_col].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1) & y.notna()
        X, y = X[mask], y[mask]

        if hyper_params is None:
            hyper_params = {
                'n_estimators': 400,
                'max_depth': 4,
                'learning_rate': 0.05,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_lambda': 1.0,
            }

        # ── Internal watch-set for per-tree guardrail ──────────────────────────
        # Stratify the 10 % hold-out by IsRegularSeason so both game types
        # are represented in the watch set.
        strat_col = source[mask]['IsRegularSeason'].astype(int)
        skf_single = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
        train_idx, watch_idx = next(skf_single.split(X, strat_col))
        X_tr,    X_watch  = X.iloc[train_idx],    X.iloc[watch_idx]
        y_tr,    y_watch  = y.iloc[train_idx],    y.iloc[watch_idx]

        # ── Custom callback: stop when val Brier < train Brier 3× in a row ────
        from xgboost.callback import TrainingCallback

        class BrierOverfitGuard(TrainingCallback):
            """
            Stops boosting if val Brier < train Brier for `patience`
            consecutive rounds (underfitting / noise regime).
            Also stops if val Brier has not improved for `early_stop` rounds
            (classic early stopping on Brier rather than logloss).
            """
            def __init__(self, X_tr, y_tr, X_val, y_val,
                         patience=3, early_stop=30, check_every=10):
                self.X_tr, self.y_tr   = X_tr, y_tr
                self.X_val, self.y_val = X_val, y_val
                self.patience          = patience
                self.early_stop        = early_stop
                self.check_every       = check_every
                self._under_count      = 0   # consecutive rounds val < train
                self._best_val         = float('inf')
                self._rounds_no_improv = 0

            def after_iteration(self, model, epoch, evals_log):
                if (epoch + 1) % self.check_every != 0:
                    return False  # don't stop

                # Booster.inplace_predict accepts numpy arrays directly
                train_prob = model.inplace_predict(self.X_tr.values)
                val_prob   = model.inplace_predict(self.X_val.values)

                train_b = brier_score_loss(self.y_tr,  train_prob)
                val_b   = brier_score_loss(self.y_val, val_prob)
                gap     = val_b - train_b

                # Classic early stopping: val Brier not improving
                if val_b < self._best_val:
                    self._best_val         = val_b
                    self._rounds_no_improv = 0
                else:
                    self._rounds_no_improv += 1
                    if self._rounds_no_improv >= self.early_stop // self.check_every:
                        print(
                            f'  [EarlyStopping] Val Brier stalled for '
                            f'{self._rounds_no_improv * self.check_every} rounds '
                            f'(val={val_b:.6f}). Stopping.'
                        )
                        return True  # stop

                # Guardrail: underfitting regime
                if val_b < train_b:
                    self._under_count += 1
                    if self._under_count >= self.patience:
                        print(
                            f'  [Guardrail] Val Brier < Train Brier for '
                            f'{self.patience} consecutive checks '
                            f'(gap={gap:+.6f}). Stopping.'
                        )
                        return True
                else:
                    self._under_count = 0

                return False  # continue training


        guard = BrierOverfitGuard(X_tr, y_tr, X_watch, y_watch,
                                  patience=3, early_stop=30, check_every=10)

        self.model = XGBClassifier(
            **hyper_params,
            random_state=42,
            eval_metric='logloss',
            verbosity=0,
            callbacks=[guard],
        )
        self.model.fit(
            X_tr, y_tr,
            eval_set=[(X_tr, y_tr), (X_watch, y_watch)],
            verbose=False,
        )
        print(f'Model trained on {"full dataset" if use_full_data else "training set"}.')

    def evaluate(self):
        """Evaluates the model against df_test."""
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')
        if self.df_test is None or self.df_test.empty:
            raise RuntimeError('No test set available. Call split_test_from_train() or get_final_test_data() first.')

        X_test = self.df_test[self.X_cols].apply(pd.to_numeric, errors='coerce')
        y_test = self.df_test[self.y_col].apply(pd.to_numeric, errors='coerce')

        mask = X_test.notna().all(axis=1) & y_test.notna()
        X_test, y_test = X_test[mask], y_test[mask]

        df_test_eval = self.df_test[mask].copy()
        df_test_eval['predicted_prob'] = self.model.predict_proba(X_test)[:, 1]

        self.brier = brier_score_loss(y_test, df_test_eval['predicted_prob'])
        print(f'Brier Score: {self.brier:.6f}')
        return self.brier

    def predict(self):
        """
        Generates March Madness win probability predictions for all hypothetical matchups.
        T1 is always the lower TeamID. Returns a dataframe with columns [ID, Pred].

        ID format: '2026_{lower_team_id}_{higher_team_id}'
        Pred: probability that the lower TeamID team wins.
        """
        if self.model is None:
            raise RuntimeError('Model has not been trained yet. Call train() first.')

        # Get all unique teams per gender from the March Madness scorecards
        df_mm = self.df_scorecards[self.df_scorecards['IsRegularSeason'] == False].copy()
        season = df_mm['Season'].max()  # predict for the most recent/current season

        # Build all hypothetical matchups within each gender
        records = []
        for is_male in [True, False]:
            teams = sorted(
                df_mm[df_mm['IsMale'] == is_male]['T1_TeamID'].tolist() +
                df_mm[df_mm['IsMale'] == is_male]['T2_TeamID'].tolist()
            )
            teams = sorted(set(teams))

            for i, t1 in enumerate(teams):
                for t2 in teams[i + 1:]:
                    records.append({
                        'IsMale': int(is_male),
                        'IsRegularSeason': 0,
                        'Season': season,
                        'T1_TeamID': t1,
                        'T2_TeamID': t2,
                    })

        df_matchups = pd.DataFrame(records)

        # Join season summary features (same logic as March Madness in _build_features)
        df_mm_features = self._compute_season_summary_features(self.df_feature_base)
        feat_cols = self.X_sing_eng_cols

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T1_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T1_{c}' for c in feat_cols})

        df_matchups = df_matchups.merge(
            df_mm_features[['Season', 'TeamID'] + feat_cols],
            left_on=['Season', 'T2_TeamID'],
            right_on=['Season', 'TeamID'],
            how='left'
        ).drop(columns=['TeamID']).rename(columns={c: f'T2_{c}' for c in feat_cols})

        X = df_matchups[self.X_cols].apply(pd.to_numeric, errors='coerce')
        mask = X.notna().all(axis=1)
        X = X[mask]
        df_matchups = df_matchups[mask].copy()

        df_matchups['Pred'] = self.model.predict_proba(X)[:, 1]
        df_matchups['ID'] = df_matchups.apply(
            lambda r: f"{int(r['Season'])}_{int(r['T1_TeamID'])}_{int(r['T2_TeamID'])}", axis=1
        )

        return df_matchups[['ID', 'Pred']].reset_index(drop=True)


### ── Phase workflows ────────────────────────────────────────────────────────────

### Phase 1 — Evaluate hyperparameters and overfitting risk
inc_xgbf = InclusiveXGBModelFiltered(df_exp_scorecards, df_exp_feature_base)
inc_xgbf.data_prep()
inc_xgbf.split_test_from_train()
xgbf_hyper_params = inc_xgbf.hyper_param_discovery_train()   # grid search → best params
inc_xgbf.train(hyper_params)                            # final train with guardrail callback
inc_xgbf.evaluate()
inc_xgbf_test_X, inc_xgbf_test_y = inc_xgbf.get_test_data()
inc_xgbf_model = inc_xgbf.get_model()
# Optional: inspect CV results table
# inc_xgb.hyper_param_results_


### Phase 2 — Evaluate against the official final test set
# inc_xgbf = InclusiveXGBModelFiltered(df_exp_scorecards, df_exp_feature_base)
# inc_xgbf.data_prep()
# inc_xgbf.get_final_test_data(df_final_test_scorecards, df_final_test_feature_base)
# inc_xgbf.train(xgbf_hyper_params)
# inc_xgbf.evaluate()

### Phase 3 — Retrain on all data and generate predictions
# inc_xgbf = InclusiveXGBModelFiltered(df_all_scorecard, df_all_features_base)
# inc_xgbf.data_prep()
# inc_xgbf.train(xgbf_hyper_params, use_full_data=True)
# inc_xgbf_prediction = inc_xgb.predict()

Data prepped.
Test set carved: 500 games.
Hyperparameter search: 144 candidates, 5-fold CV each.
  [   1/144] val=0.194220  train=0.192025  gap=+0.002195 ← best so far
  [   2/144] val=0.194206  train=0.192035  gap=+0.002171 ← best so far
  [   3/144] val=0.194022  train=0.191878  gap=+0.002144 ← best so far
  [   4/144] val=0.194057  train=0.191904  gap=+0.002153
  [   5/144] val=0.194294  train=0.192014  gap=+0.002280
  [   6/144] val=0.194255  train=0.192010  gap=+0.002244
  [   7/144] val=0.194199  train=0.191878  gap=+0.002321
  [   8/144] val=0.194160  train=0.191890  gap=+0.002270
  [   9/144] val=0.210351  train=0.209588  gap=+0.000762
  [  10/144] val=0.210369  train=0.209604  gap=+0.000765
  [  11/144] val=0.210107  train=0.209350  gap=+0.000757
  [  12/144] val=0.210100  train=0.209358  gap=+0.000742
  [  13/144] val=0.210441  train=0.209633  gap=+0.000808
  [  14/144] val=0.210444  train=0.209632  gap=+0.000812
  [  15/144] val=0.210228  train=0.209410  gap=+0.000818
  [  1

## Combined Experiment

In [278]:
xgbf_pred = inc_xgbf_model.predict_proba(inc_xgbf_test_X)
log_pred = inc_log_model.predict_proba(inc_log_test_X)

comb = np.mean( [log_pred,xgbf_pred], axis=0)

brier_score_loss(inc_xgbf_test_y, comb)

0.19765234224806583